# Capítulo 23 · Laboratório 23: price action, Al Brooks e Linda Raschke
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Regras públicas e simplificadas, escritas para o teste: não reproduzem o olho de Al Brooks nem as regras completas de Linda Raschke. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap23.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O Yahoo só guarda ~730 pregões de barras de 60 minutos e ~60 pregões de barras de 5 minutos, contados de hoje. As barras do livro vão de 23/10/2024 (60 minutos) e de 17/08/2026 (5 minutos) até 30/09/2026: quando esse período sai da janela, o notebook usa o que o Yahoo ainda guarda e avisa, e as datas e os números das Partes 6 e 7 mudam. O laboratório conta as barras 1/2/3, testa pin bar, inside bar, engolfo e fakey a favor e contra a tendência, a segunda entrada (H1 e H2), o rompimento forte, as frases de Brooks e os setups de Raschke.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
INI60, INI5 = "2024-10-23", "2026-08-16"  # início das barras de 60 minutos e das de 5 minutos do livro
CACHE = "dados_cap23.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


pc = lambda x, k=0: "" if pd.isna(x) else f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=2: "" if pd.isna(x) else ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:+,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
nn = lambda x: f"{x:,}".replace(",", ".")
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
INTRA = ["SPY", "QQQ", "IWM", "DIA", "BOVA11.SA", "PETR4.SA", "VALE3.SA", "ITUB4.SA"]

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):
    D, DX, H60, M5 = pd.read_pickle(CACHE)
else:
    # O Yahoo só aceita data de início nos últimos ~730 dias (60 minutos) e ~60 dias (5 minutos). Pedindo o
    # período máximo (period), ele devolve ~730 e ~60 pregões; o corte refaz o período do livro enquanto ele couber.
    def intra(tickers, periodo, intervalo, ini):
        x = yf.download(tickers, period=periodo, interval=intervalo, auto_adjust=True, progress=False,
                        group_by="ticker")
        confere({f"as barras de {intervalo[:-1]} minutos": x})
        dia = x.index.strftime("%Y-%m-%d")
        corte = x[(dia >= ini) & (dia < FIM)]
        return corte if len(corte) else x       # o período do livro saiu da janela: fica a mais recente

    D = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    DX = yf.download([t + ".SA" for t in B3], start="2022-01-01", end=FIM, auto_adjust=False, progress=False,
                     group_by="ticker")                       # exemplos: preços da época, como na plataforma
    H60 = intra(INTRA, "730d", "60m", INI60)
    M5 = intra(INTRA, "60d", "5m", INI5)
    confere({"as 40 ações": D, "as ações dos exemplos": DX})
    if (f"{H60.index[0]:%Y-%m-%d}" > "2024-10-23" or f"{M5.index[0]:%Y-%m-%d}" > "2026-08-17"
            or f"{H60.index[-1]:%Y-%m-%d}" != "2026-09-30" or f"{M5.index[-1]:%Y-%m-%d}" != "2026-09-30"):
        print(f"Aviso: o Yahoo só guarda ~730 pregões de barras de 60 minutos e ~60 de barras de 5 minutos, contados de"
              f" hoje. Esta execução usa a janela que o Yahoo guarda hoje, e não a do livro (60 minutos de 23/10/2024 a"
              f" 30/09/2026; 5 minutos de 17/08/2026 a 30/09/2026): 60 minutos de {H60.index[0]:%d/%m/%Y} a"
              f" {H60.index[-1]:%d/%m/%Y} e 5 minutos de {M5.index[0]:%d/%m/%Y} a {M5.index[-1]:%d/%m/%Y}. As datas e os"
              " números das Partes 6 e 7 diferem dos do livro.")
    pd.to_pickle((D, DX, H60, M5), CACHE)


def limpa(d):
    """Sem filtro de volume: o Yahoo informa volume zero em algumas barras intradiárias da B3 (a das 10h)."""
    d = d.drop(columns=["Adj Close"], errors="ignore").dropna()
    return d[(d["Close"] > 0) & (d["High"] > d["Low"])]


def atr(d, n=14):
    h, l, c = d["High"], d["Low"], d["Close"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean()


def adx(d, n=14):
    """ADX de Wilder, com o +DI e o -DI."""
    h, l, c = d["High"], d["Low"], d["Close"]
    up, dn = h.diff(), -l.diff()
    pdm = np.where((up > dn) & (up > 0), up, 0.0)
    ndm = np.where((dn > up) & (dn > 0), dn, 0.0)
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    a = 1 / n
    atr_w = tr.ewm(alpha=a, adjust=False).mean()
    pdi = 100 * pd.Series(pdm, index=d.index).ewm(alpha=a, adjust=False).mean() / atr_w
    ndi = 100 * pd.Series(ndm, index=d.index).ewm(alpha=a, adjust=False).mean() / atr_w
    dx = 100 * (pdi - ndi).abs() / (pdi + ndi)
    return dx.ewm(alpha=a, adjust=False).mean(), pdi, ndi


def resultado_r(h, l, c, i, entrada, stop, alvo_r=2.0, prazo=20):
    """Resultado em R de uma compra: alvo de alvo_r R, stop de 1R; se os dois na mesma barra, conta o stop."""
    risco = entrada - stop
    if risco <= 0:
        return np.nan
    alvo = entrada + alvo_r * risco
    fim = min(i + prazo, len(c) - 1)
    for k in range(i, fim + 1):
        if l[k] <= stop:
            return -1.0
        if h[k] >= alvo:
            return alvo_r
    return (c[fim] - entrada) / risco


def sinais(d):
    """Todos os sinais de alta do capítulo para uma série diária."""
    o, h, l, c = (d[k] for k in ["Open", "High", "Low", "Close"])
    a14 = atr(d)
    ema = c.ewm(span=20, adjust=False).mean()
    alta = (c > ema) & (ema > ema.shift(5))                       # contexto: acima da média de 20 subindo
    baixa = (c < ema) & (ema < ema.shift(5))
    amp = h - l
    interna = (h <= h.shift()) & (l >= l.shift())
    externa = (h > h.shift()) & (l < l.shift())
    pin = ((np.minimum(o, c) - l) >= 2 / 3 * amp) & (amp >= a14) & (l <= l.rolling(10).min())
    ib = interna.shift(1, fill_value=False)
    rompe_ib = ib & (h > h.shift())                               # hoje rompe a máxima da inside bar de ontem
    engolfo = externa & (c > o) & (c > h.shift()) & (l <= l.rolling(10).min())
    fakey = ib & (l < l.shift()) & (c > l.shift()) & (c < h.shift(2)) & (c > l.shift(2))
    min20 = l.rolling(20).min().shift(1)
    idade = l.rolling(20).apply(np.argmin, raw=True).shift(1)     # posição da mínima dentro da janela
    soup = (l < min20) & (c > min20) & (idade <= 15)               # mínima anterior com pelo menos 4 barras
    ad, pdi, ndi = adx(d)
    # ADX acima de 30 e subindo, com o +DI acima do -DI (a força é da alta, não de uma queda anterior)
    graal = (ad > 30) & (ad > ad.shift()) & (pdi > ndi) & (l <= ema) & (c > ema) & alta
    abriu_topo = (o.shift() - l.shift()) >= 0.8 * amp.shift()
    fechou_fundo = (c.shift() - l.shift()) <= 0.2 * amp.shift()
    oitenta = abriu_topo & fechou_fundo & (l < l.shift()) & (c > l.shift())
    return dict(o=o, h=h, l=l, c=c, ema=ema, alta=alta, baixa=baixa, amp=amp, interna=interna, externa=externa,
                pin=pin, rompe_ib=rompe_ib, engolfo=engolfo, fakey=fakey, soup=soup, graal=graal, oitenta=oitenta,
                min20=min20, adx=ad)


def segundas_entradas(s):
    """H1 e H2 de Al Brooks numa tendência de alta: conta as tentativas de retomar a alta durante o recuo.
    Devolve (índice da barra de entrada, tipo, resultado em R)."""
    H, L, C, A, O = (s[k].to_numpy() for k in ("h", "l", "c", "alta", "o"))
    out = []
    topo, recuo, conta, desceu = H[0], False, 0, False
    for i in range(1, len(H) - 1):
        if H[i] > topo and not recuo:
            topo = H[i]
            continue
        if not recuo and H[i] < H[i - 1]:
            recuo, conta, desceu = True, 0, True
            continue
        if recuo:
            if H[i] > H[i - 1] and desceu:
                conta += 1
                desceu = False
                if conta <= 2 and A[i - 1]:                         # barra de sinal = a anterior
                    entrada = max(H[i - 1], O[i])                       # se abriu acima da ordem, compra na abertura
                    out.append((i, f"H{conta}", resultado_r(H, L, C, i, entrada, L[i - 1])))
            elif H[i] <= H[i - 1]:
                desceu = True
            if H[i] > topo:                                         # nova máxima: acabou o recuo
                topo, recuo = H[i], False
    return out


def rompimentos_fortes(s):
    """Duas barras grandes de alta fechando no quarto superior, acima do topo de 20 barras. Alvo: movimento medido
    (altura da faixa de 20 barras somada ao topo); stop: mínima da primeira barra."""
    o, h, l, c, amp = s["o"], s["h"], s["l"], s["c"], s["amp"]
    topo20, fundo20 = h.rolling(20).max().shift(2), l.rolling(20).min().shift(2)
    grande = ((c - o) > 1.5 * (c - o).abs().rolling(20).mean()) & ((c - l) >= 0.75 * amp)
    forte = grande & grande.shift(1, fill_value=False) & (c.shift() > topo20) & (o > topo20) & (c > topo20)
    H, L = h.to_numpy(), l.to_numpy()
    out = []
    for i in np.flatnonzero(forte.to_numpy()):
        alvo = topo20.iloc[i] + (topo20.iloc[i] - fundo20.iloc[i])
        stop = L[i - 1]
        res = "nem um nem outro em 60 barras"
        for k in range(i + 1, min(i + 61, len(H))):
            if L[k] <= stop:
                res = "stop (mínima da 1ª barra) antes"
                break
            if H[k] >= alvo:
                res = "movimento medido antes"
                break
        out.append((i, res, alvo, stop, topo20.iloc[i], fundo20.iloc[i]))
    return out


tipos, padroes, segunda, fortes, raschke, olhando = [], [], [], [], [], []
for t in TICKERS:
    d = limpa(D[t])
    if len(d) < 500:
        continue
    s = sinais(d)
    c = s["c"]
    f5, f10 = c.shift(-5) / c - 1, c.shift(-10) / c - 1
    tipos.append(pd.Series(np.select([s["interna"], s["externa"]], ["1 (inside)", "3 (outside)"],
                                     "2 (direcional)"))[1:])
    for nome, chave in [("pin bar de alta (fundo de 10 barras)", "pin"), ("rompimento da inside bar para cima", "rompe_ib"),
                        ("engolfo de alta (fundo de 10 barras)", "engolfo"), ("fakey de alta", "fakey")]:
        for ctx_nome, ctx in [("a favor (tendência de alta)", s["alta"]), ("contra (tendência de baixa)", s["baixa"])]:
            m = s[chave] & ctx
            padroes.append(pd.DataFrame({"padrao": nome, "contexto": ctx_nome, "f5": f5[m], "f10": f10[m]}))
    padroes.append(pd.DataFrame({"padrao": "qualquer barra", "contexto": "todas", "f5": f5, "f10": f10}))
    segunda += [{"tipo": tp, "r": r} for _, tp, r in segundas_entradas(s)]
    fortes += [x[1] for x in rompimentos_fortes(s)]
    for nome, chave in [("Turtle Soup", "soup"), ("Holy Grail", "graal"), ("80-20", "oitenta")]:
        m = s[chave]
        raschke.append(pd.DataFrame({"setup": nome, "f5": f5[m], "f10": f10[m]}))
    # a primeira versão do notebook: comprava no nível exato (a mínima de 20 dias) só nos dias que fecharam acima
    olhando.append((c.shift(-10) / s["min20"] - 1)[s["soup"]])

## Parte 2: barras 1, 2 e 3

In [ ]:
print(f"Parte 1. {len(TICKERS)} ações de 2005 a 30/09/2026; 60 minutos de {limpa(H60['SPY']).index[0]:%d/%m/%Y};"
      f" 5 minutos de {limpa(M5['SPY']).index[0]:%d/%m/%Y}.")
TP = pd.concat(tipos).value_counts(normalize=True)
print("\nParte 2. Barras 1/2/3 no diário (40 ações desde 2005)")
print("   " + " | ".join(f"{k}: {pc(v)}" for k, v in TP.items()))

## Parte 3: padrões clássicos

In [ ]:
P = pd.concat(padroes).dropna()
print("\nParte 3. Padrões clássicos: retorno médio (casos positivos) depois do sinal")
print("   padrão                                  contexto                       casos |   5 barras     |  10 barras")
P3 = {}
for (pad, ctx), x in P.groupby(["padrao", "contexto"], sort=False):
    P3[(pad, ctx)] = (len(x), x["f10"].mean(), (x["f10"] > 0).mean())
    print(f"   {pad:<39} {ctx:<28} {nn(len(x)):>7} | {ps(x['f5'].mean())} ({pc((x['f5'] > 0).mean())}) |"
          f" {ps(x['f10'].mean())} ({pc((x['f10'] > 0).mean())})")

In [ ]:
import matplotlib.pyplot as plt
P3t = P[P["padrao"] != "qualquer barra"].groupby(["padrao", "contexto"])["f10"].mean().unstack()
ax = P3t.plot(kind="bar", figsize=(9, 4), color=["white", "gray"], edgecolor="black", rot=0,
              title="Retorno médio em 10 barras: a favor e contra a tendência")
ax.axhline(P[P["padrao"] == "qualquer barra"]["f10"].mean(), color="black", ls="--", label="barra qualquer")
ax.legend()
plt.show()

## Parte 4: segunda entrada

In [ ]:
S = pd.DataFrame(segunda).dropna()
print("\nParte 4. Segunda entrada de Al Brooks em tendência de alta (alvo 2R, stop 1R, até 20 barras)")
P4 = {}
for k, x in S.groupby("tipo"):
    P4[k] = (len(x), x["r"].mean(), (x["r"] == 2).mean(), (x["r"] == -1).mean())
    print(f"   {k}: {nn(len(x)):>6} operações | resultado médio {nr(x['r'].mean())} R | alvo {pc((x['r'] == 2).mean())}"
          f" | stop {pc((x['r'] == -1).mean())}")

## Parte 5: rompimento forte

In [ ]:
F = pd.Series(fortes).value_counts(normalize=True)
print(f"\nParte 5. Rompimento forte (duas barras grandes de alta acima do topo de 20 barras): {nn(len(fortes))} casos")
for k, v in F.items():
    print(f"   {k}: {pc(v)}")

## Parte 6: 40% a 60% no gráfico de 5 minutos

In [ ]:
tot = sobe = forte_t = forte_s = fraca_t = fraca_s = 0
for t in INTRA:
    x = limpa(M5[t])
    a = atr(x, 20).to_numpy()
    H, L, C, O = (x[k].to_numpy() for k in ["High", "Low", "Close", "Open"])
    amp = H - L
    media_amp = pd.Series(amp).rolling(20).mean().to_numpy()
    touro = (C > O) & ((C - L) >= 0.75 * amp) & (amp > media_amp)
    urso = (C < O) & ((H - C) >= 0.75 * amp) & (amp > media_amp)
    for i in range(22, len(x) - 24):
        if np.isnan(a[i]) or x.index[i].date() != x.index[i + 24].date():
            continue                                          # só dentro do mesmo pregão
        cima, baixo = C[i] + a[i], C[i] - a[i]
        res = None
        for k in range(i + 1, i + 25):
            u, dd = H[k] >= cima, L[k] <= baixo
            if u and dd:
                break
            if u or dd:
                res = u
                break
        if res is None:
            continue
        tot += 1
        sobe += res
        if touro[i] and touro[i - 1] and touro[i - 2]:
            forte_t += 1
            forte_s += res
        if urso[i] and urso[i - 1] and urso[i - 2]:
            fraca_t += 1
            fraca_s += res
P6 = (tot, sobe / tot, forte_t, forte_s / max(forte_t, 1), fraca_t, 1 - fraca_s / max(fraca_t, 1))
print("\nParte 6. Gráfico de 5 minutos (8 ativos, ~60 dias): subir 1 ATR antes de cair 1 ATR, no mesmo pregão")
print(f"   qualquer barra: {pc(P6[1])} de {nn(tot)} casos")
print(f"   depois de 3 barras fortes de alta seguidas: {pc(P6[3])} sobem primeiro, de {nn(forte_t)} casos"
      f" ({pc(forte_t / tot, 1)} das barras)")
print(f"   depois de 3 barras fortes de baixa seguidas: {pc(P6[5])} caem primeiro, de {nn(fraca_t)} casos"
      f" ({pc(fraca_t / tot, 1)} das barras)")

## Parte 7: Linda Raschke

In [ ]:
R = pd.concat(raschke).dropna()
print("\nParte 7. Linda Raschke, versões simplificadas (diário; entrada no fechamento do dia do sinal)")
P7 = {}
for k, x in R.groupby("setup", sort=False):
    P7[k] = (len(x), x["f10"].mean(), (x["f10"] > 0).mean())
    print(f"   {k:<12} {nn(len(x)):>6} casos | 5 barras {ps(x['f5'].mean())} ({pc((x['f5'] > 0).mean())}) |"
          f" 10 barras {ps(x['f10'].mean())} ({pc((x['f10'] > 0).mean())})")
base = P[P["padrao"] == "qualquer barra"]
print(f"   qualquer barra (referência): 5 barras {ps(base['f5'].mean())} | 10 barras {ps(base['f10'].mean())}")
OL = pd.concat(olhando).dropna()
print(f"   viés de olhar o futuro: o mesmo Turtle Soup comprando no nível da mínima de 20 dias, só nos dias que fecharam"
      f" acima dela: 10 barras {ps(OL.mean())} ({pc((OL > 0).mean())})")

pin_r = []
for t in INTRA:
    x = limpa(H60[t])
    dias = x.groupby(x.index.date)
    fech = dias["Close"].last()
    roc = fech.pct_change()
    ganho, perda = roc.diff().clip(lower=0), -roc.diff().clip(upper=0)
    lbr = 100 - 100 / (1 + ganho.ewm(alpha=1 / 3, adjust=False).mean() / perda.ewm(alpha=1 / 3, adjust=False).mean())
    datas = list(fech.index)
    for j in range(1, len(datas)):
        if lbr.iloc[j - 1] >= 30:
            continue
        dia = dias.get_group(datas[j])
        if len(dia) < 3:
            continue
        ent, stp = dia["High"].iloc[0], dia["Low"].iloc[0]
        resto = dia.iloc[1:]
        rompeu = resto["High"].to_numpy() > ent
        if not rompeu.any():
            continue
        k = int(np.argmax(rompeu))
        depois = resto.iloc[k:]
        pin_r.append(-1.0 if (depois["Low"] <= stp).any() else (dia["Close"].iloc[-1] - ent) / (ent - stp))
PR = np.array(pin_r)
print(f"   Momentum Pinball (60 minutos, 8 ativos, ~2 anos; compra acima da primeira hora quando o LBR/RSI está abaixo"
      f" de 30; stop na mínima da primeira hora; saída no fechamento): {nn(len(PR))} operações | {nr(PR.mean())} R |"
      f" acerto {pc((PR > 0).mean())}")

## Parte 8: exemplos reais na B3 (preços da época)

In [ ]:
EX = {"fakey": [], "H2": [], "forte": [], "graal": []}
for t in [t + ".SA" for t in B3]:
    d = limpa(DX[t])
    s = sinais(d)
    c = s["c"]
    f10 = c.shift(-10) / c - 1
    ok = d.index >= "2023-01-01"
    for i in np.flatnonzero((s["fakey"] & s["alta"]).to_numpy() & ok):
        EX["fakey"].append((t, d.index[i], f10.iloc[i]))
    for i in np.flatnonzero(s["graal"].to_numpy() & ok):
        EX["graal"].append((t, d.index[i], f10.iloc[i]))
    for i, tp, r in segundas_entradas(s):
        if tp == "H2" and ok[i]:
            EX["H2"].append((t, d.index[i], r))
    for i, res, alvo, stop, topo, fundo in rompimentos_fortes(s):
        if ok[i]:
            EX["forte"].append((t, d.index[i], res, alvo, stop, topo, fundo))
print("\nParte 8. Exemplos reais na B3 desde 2023 (preços da época), para as figuras")
for k in ("fakey", "graal", "H2"):
    x = pd.DataFrame(EX[k], columns=["ação", "data", "resultado"]).dropna()
    print(f"   {k}: {len(x)} casos | melhores: " + "; ".join(f"{a[:-3]} {dt:%d/%m/%Y} ({v:+.2f})"
          for a, dt, v in x.sort_values("resultado", ascending=False).head(4).itertuples(index=False)))
xf = pd.DataFrame(EX["forte"], columns=["ação", "data", "resultado", "alvo", "stop", "topo", "fundo"])
print(f"   rompimento forte: {len(xf)} casos | " + "; ".join(f"{a[:-3]} {dt:%d/%m/%Y} ({r})"
      for a, dt, r in xf[["ação", "data", "resultado"]].head(8).itertuples(index=False)))

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, o trecho do Turtle Soup na função `sinais` da Parte 1 (`min20`, `idade` e `soup`) e a linha que monta a lista `olhando`. Mostre por que comprar no nível da mínima de 20 dias, só nos dias que fecharam acima dela, usa uma informação que não existia na hora da entrada." Confira com a Parte 7, que imprime o Turtle Soup com entrada no fechamento (+0,30% em 10 pregões, no livro) e a versão com o viés de olhar o futuro (+1,49%).
2. **Adaptar:** "Numa célula nova depois da Parte 4, baixe o diário de [ATIVO] do Yahoo Finance desde 2005, com `auto_adjust=True` como na Parte 1, passe por `limpa` e `sinais` e rode `segundas_entradas`. Imprima o número de operações e o resultado médio em R do H1 e do H2, como na Parte 4. Não mude o contexto de alta, o stop nem o alvo de 2R." Antes de olhar o ativo novo, confira se a Parte 4 mostra os números do livro: H1 com 0,00 R em 3.696 operações e H2 com −0,02 R em 1.224. Um ativo só é amostra pequena: qualquer conclusão precisa do teste fora da amostra (Capítulo 27).

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 3, troque o contexto (média de 20) por média de 50. Os padrões a favor melhoram?
2. Na Parte 4, use alvo de 1R em vez de 2R. Qual o acerto do H2?
3. Na Parte 5, exija três barras grandes em vez de duas. O movimento medido chega mais vezes?
4. Na Parte 7, compre o Turtle Soup no dia seguinte, acima da máxima do dia do sinal. O resultado muda?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.